# Lab 02 — Predict, measure, explain an inference bottleneck

**Outcome:** a reproducible phase experiment, a matched batch comparison and two focused traces. Explain one prediction that did not survive measurement. This is not an HTTP benchmark or a quality test.

Read [Book 02](https://levshaazz.github.io/llm-serving-mastery/en/book/02/) and the [student guide](https://levshaazz.github.io/llm-serving-mastery/en/start/). You independently arrange a CUDA GPU (Linux/Colab/WSL; 16 GiB recommended). CPU contract checks do not replace the GPU run. Use your private repository's GitHub Issue with @levshaazz for diagnostics; do not post secrets or private artifacts publicly.

**Protocol v2:** two pinned models × five cases × five raw trials = **50 trials**. Cases are batch 1 × input {128,4096} × output {32,128}, plus batch 4/input 128/output 32. Batch 1/input 128/output 32 is the matched batch control. Output 512 and batch 8 are optional follow-ups only, after the core passes. Historical v1 results are not a v2 baseline.

**Before execution:** write three predictions: which phase changes with prompt length, which with output length, and whether batch 4 raises aggregate rate while changing per-sequence cadence. Name the measurement that could disprove each prediction.

Run one notebook/process at a time. Models reuse a persistent cache; execution artifacts get new directories. Stop on OOM, resource growth or a gate failure; preserve the partial JSON. Do not restart the entire matrix blindly.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess, sys
pinned = {'transformers': '4.51.3', 'accelerate': '1.6.0', 'huggingface_hub': '0.30.2', 'safetensors': '0.5.3'}
for name, wanted in pinned.items():
    try: installed = version(name) == wanted
    except PackageNotFoundError: installed = False
    if not installed:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'{name}=={wanted}'])
# If you previously imported a different Transformers version, restart the kernel now.


## 1. Read the measurement boundary

The following cell is the complete editable implementation. Prefill includes one forward and selection of output y1; decode includes O−1 forwards and greedy selections. The last selected token has no KV until it is fed back, so retained length is P+O−1. EOS is ignored to hold lengths fixed. Inputs are repeated synthetic token IDs, with no padding or chat-template claim.

Each phase records synchronized wall time and a CUDA-event interval. **An event interval is not a sum of kernel durations:** host launch gaps can lie between events. Neither timer includes tokenization, model loading, HTTP, queueing or delivery. Do not call these values server TTFT/TPOT or p95.

Last-position logits avoid computing unused vocabulary projections for every prompt position. Full-shape warmup is outside timing; cases are shuffled within five rounds, one model at a time. Allocator pools stay warm across trials. Post-trial live-allocation recovery and host RAM are checked; peaks are recorded. Gates are stop rules, not an OS memory sandbox. Instructor runs also use an external memory limit and wall timeout.


In [ ]:
"""Topic 02 protocol v2. Embedded verbatim in the standalone public notebook.

No downloads on import; no CUDA required for the arithmetic/contract functions.
This is a guarded Python phase experiment, not an HTTP service benchmark.
"""
import gc
import json
import math
import os
import platform
import random
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

PROTOCOL = "topic02-phase-v2"
MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]
TRIALS = 5
ORDER_SEED = 20260930
GIB = 1024 ** 3


def kv_bytes_per_token(config, element_bytes=2):
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    return 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * element_bytes


def attention_pairs(prompt, output):
    """Decode only: output 1 is selected from prefill; each later query includes itself."""
    steps = output - 1
    return steps * prompt + steps * (steps + 1) // 2


def trial_order(cases=CASES, trials=TRIALS):
    rng = random.Random(ORDER_SEED)
    schedule = []
    for repeat in range(trials):
        order = list(cases)
        rng.shuffle(order)
        schedule.extend((repeat, *case) for case in order)
    return schedule


def checkpoint(path, data):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def host_available_bytes():
    # Colab and the instructor's WSL environment are Linux. Fail closed on unknown hosts.
    meminfo = Path("/proc/meminfo")
    if not meminfo.exists():
        raise RuntimeError("Run this CUDA lab on Linux/Colab/WSL; host memory gate is unavailable")
    for line in meminfo.read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("MemAvailable is unavailable")


def process_rss_bytes():
    return int(Path("/proc/self/statm").read_text().split()[1]) * os.sysconf("SC_PAGE_SIZE")


def resource_gate(torch, baseline_allocated=None, baseline_rss=None):
    allocated = torch.cuda.memory_allocated()
    free, total = torch.cuda.mem_get_info()
    limit = min(12 * GIB, int(total * 0.80))
    if allocated > limit or total - free > min(12 * GIB, int(total * 0.90)):
        raise RuntimeError("STOP: GPU memory budget exceeded; retain partial evidence and inspect other processes")
    if host_available_bytes() < 2 * GIB:
        raise RuntimeError("STOP: less than 2 GiB host RAM available")
    if baseline_allocated is not None and allocated > baseline_allocated + 128 * 1024 ** 2:
        raise RuntimeError("STOP: live CUDA allocations failed to return to the post-load baseline")
    if baseline_rss is not None and process_rss_bytes() > baseline_rss + 2 * GIB:
        raise RuntimeError("STOP: process RSS grew by more than 2 GiB")
    return {"allocated_bytes": allocated, "reserved_bytes": torch.cuda.memory_reserved(),
            "device_used_bytes": total - free, "host_available_bytes": host_available_bytes(),
            "process_rss_bytes": process_rss_bytes()}


def exact_input(torch, tokenizer, batch, length):
    tokens = tokenizer("Explain GPU memory reuse and token dependencies. ", add_special_tokens=False)["input_ids"]
    row = (tokens * math.ceil(length / len(tokens)))[:length]
    return torch.tensor([row] * batch, dtype=torch.long, device="cuda")


def phase_trial(torch, model, ids, output_tokens):
    """No HTTP, tokenization, model load or profiler inside these intervals.

    Prefill includes selecting y1. Decode includes O-1 forwards and greedy selections.
    CUDA-event elapsed time spans host launch gaps too; it is not summed kernel time.
    Exact-length synthetic generation deliberately ignores EOS; it is not a quality test.
    """
    out = past = token = None
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    try:
        with torch.inference_mode():
            wall_start = time.perf_counter()
            start.record()
            out = model(ids, use_cache=True, logits_to_keep=1)
            token = out.logits[:, -1].argmax(-1, keepdim=True)
            past = out.past_key_values
            out = None  # Do not retain a full prompt-logit tensor during decode.
            end.record()
            end.synchronize()
            prefill_wall_ms = (time.perf_counter() - wall_start) * 1000
            prefill_event_ms = start.elapsed_time(end)
            prefill_peak = torch.cuda.max_memory_allocated()
            wall_start = time.perf_counter()
            start.record()
            for _ in range(output_tokens - 1):
                out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                token = out.logits[:, -1].argmax(-1, keepdim=True)
                past = out.past_key_values
                out = None
            end.record()
            end.synchronize()
            decode_wall_ms = (time.perf_counter() - wall_start) * 1000
            decode_event_ms = start.elapsed_time(end)
            retained = int(past.get_seq_length())
            expected = ids.shape[1] + output_tokens - 1
            if retained != expected:
                raise AssertionError(f"KV sequence length {retained} != {expected}")
            steps = output_tokens - 1
            return {
                "prefill_wall_ms": prefill_wall_ms, "prefill_event_ms": prefill_event_ms,
                "decode_wall_ms": decode_wall_ms, "decode_event_ms": decode_event_ms,
                "decode_steps_per_sequence": steps, "generated_tokens": ids.shape[0] * output_tokens,
                "retained_tokens_per_sequence": retained,
                "decode_aggregate_tokens_per_s": ids.shape[0] * steps * 1000 / decode_wall_ms,
                "decode_mean_step_ms": decode_wall_ms / steps,
                "prefill_peak_allocated_bytes": prefill_peak,
                "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
            }
    finally:
        out = past = token = None


def run_matrix(model_paths, output_dir, *, smoke=False):
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required; CPU checks cannot replace this run")
    if transformers.__version__ != "4.51.3":
        raise RuntimeError("Use the pinned Transformers 4.51.3 runtime")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)  # Never silently overwrite a previous attempt.
    output_path = output_dir / "topic02-benchmark.json"
    cases = [CASES[0]] if smoke else CASES
    model_ids = list(MODELS)[:1] if smoke else list(MODELS)
    data = {"protocol": PROTOCOL, "status": "running", "scope": "smoke" if smoke else "core",
            "started_at": datetime.now(timezone.utc).isoformat(),
            "environment": {"python": platform.python_version(), "torch": torch.__version__,
                            "transformers": transformers.__version__, "cuda": torch.version.cuda,
                            "gpu": torch.cuda.get_device_name(), "dtype": "float16",
                            "attention_requested": "sdpa", "logits_to_keep": 1},
            "models": MODELS, "order_seed": ORDER_SEED, "trials": 1 if smoke else TRIALS,
            "cases": [list(c) for c in cases], "results": [], "resource_samples": [],
            "measurement": "synchronized wall + CUDA-event intervals, greedy forced length, no HTTP",
            "model_details": {}}
    checkpoint(output_path, data)
    model = tokenizer = ids = None
    try:
        data["resource_samples"].append(resource_gate(torch))
        for model_id in model_ids:
            try:
                model = AutoModelForCausalLM.from_pretrained(
                    model_paths[model_id], local_files_only=True, torch_dtype=torch.float16,
                    attn_implementation="sdpa").to("cuda").eval()
                tokenizer = AutoTokenizer.from_pretrained(model_paths[model_id], local_files_only=True)
                torch.cuda.synchronize()
                baseline = torch.cuda.memory_allocated()
                rss = process_rss_bytes()
                data["model_details"][model_id] = {
                    "kv_bytes_per_token": kv_bytes_per_token(model.config),
                    "post_load_allocated_bytes": baseline,
                    "attention_selected": model.config._attn_implementation,
                    "vocab_size": model.config.vocab_size,
                }
                resource_gate(torch, baseline, rss)
                # Warm each actual shape, including the complete decode path. Warmup is not a trial.
                for batch, prompt, output in cases:
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    resource_gate(torch, baseline, rss)
                for repeat, batch, prompt, output in trial_order(cases, data["trials"]):
                    resource_gate(torch, baseline, rss)
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    result = phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    result.update(model=model_id, batch=batch, input_tokens=prompt,
                                  output_tokens=output, repeat=repeat)
                    data["results"].append(result)
                    checkpoint(output_path, data)  # Preserve the trial before checking recovery.
                    data["resource_samples"].append(resource_gate(torch, baseline, rss))
                    if result["peak_allocated_bytes"] > min(12 * GIB, torch.cuda.get_device_properties(0).total_memory * .80):
                        raise RuntimeError("STOP: an in-process allocation peak exceeded the budget")
                    checkpoint(output_path, data)
                    print(model_id, (batch, prompt, output), "trial", repeat + 1, flush=True)
            finally:
                model = tokenizer = ids = None
                gc.collect()
                torch.cuda.empty_cache()  # Between models only, never between timed trials.
                torch.cuda.synchronize()
                data["resource_samples"].append(resource_gate(torch))
        data["status"] = "complete"
    except Exception as exc:
        data["status"] = "failed"
        data["error_type"] = type(exc).__name__
        raise
    finally:
        data["finished_at"] = datetime.now(timezone.utc).isoformat()
        checkpoint(output_path, data)
    return data


def summarize(data):
    rows = []
    groups = {}
    for trial in data["results"]:
        key = tuple(trial[k] for k in ("model", "batch", "input_tokens", "output_tokens"))
        groups.setdefault(key, []).append(trial)
    for key, trials in groups.items():
        row = dict(zip(("model", "batch", "input_tokens", "output_tokens"), key))
        row["n"] = len(trials)
        for metric in ("prefill_wall_ms", "decode_wall_ms", "decode_mean_step_ms", "decode_aggregate_tokens_per_s"):
            values = [t[metric] for t in trials]
            row[metric] = {"median": statistics.median(values), "min": min(values), "max": max(values)}
        rows.append(row)
    return rows


def focused_profiles(model_path, output_dir):
    """Two independent traces: P=128 and P=4096, each prefill + seven decode steps.

    No profile_memory/record_shapes by default: these add overhead/retain references.
    The trace proves event ordering/operator attribution, not unprofiled latency or HBM bandwidth.
    """
    import torch
    from torch.profiler import profile, record_function, ProfilerActivity
    from transformers import AutoModelForCausalLM, AutoTokenizer
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    model = tokenizer = ids = out = past = token = None
    summaries = []
    try:
        resource_gate(torch)
        model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True,
                    torch_dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        baseline = torch.cuda.memory_allocated()
        for prompt in (128, 4096):
            ids = exact_input(torch, tokenizer, 1, prompt)
            phase_trial(torch, model, ids, 8)
            with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                with record_function("topic02_prefill"):
                    out = model(ids, use_cache=True, logits_to_keep=1)
                    past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                    out = None
                with record_function("topic02_decode_7_steps"):
                    for _ in range(7):
                        out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                        past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                        out = None
                torch.cuda.synchronize()
            prof.export_chrome_trace(str(output_dir / f"topic02-trace-p{prompt}.json"))
            events = sorted(prof.key_averages(), key=lambda e: e.self_device_time_total, reverse=True)[:12]
            summaries.append({"input_tokens": prompt, "output_tokens": 8, "model": list(MODELS)[0],
                              "operators": [{"name": e.key, "calls": e.count,
                                             "self_device_us": e.self_device_time_total} for e in events]})
            ids = out = past = token = prof = events = None
            gc.collect()
            resource_gate(torch, baseline)
            checkpoint(output_dir / "topic02-profile-summary.json", summaries)
    finally:
        model = tokenizer = ids = out = past = token = None
        gc.collect()
        torch.cuda.empty_cache()
    return summaries


## 2. Resolve pinned models once

Set **LSM_MODEL_CACHE** to persistent storage before running this cell (for example a mounted persistent volume). The default cache is persistent only while this runtime survives. An offline instructor run sets HF_HUB_OFFLINE=1; missing snapshots then fail without downloading. Never delete model caches when making a fresh evidence directory.


In [ ]:
from huggingface_hub import snapshot_download
cache = Path(os.environ.get('LSM_MODEL_CACHE', str(Path.home() / '.cache' / 'lsm-models')))
cache.mkdir(parents=True, exist_ok=True)
MODEL_PATHS = {}
for model_id, revision in MODELS.items():
    try:
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache, local_files_only=True)
    except FileNotFoundError:
        if os.environ.get('HF_HUB_OFFLINE') == '1':
            raise RuntimeError('Pinned model is missing from the offline cache')
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache)
    MODEL_PATHS[model_id] = resolved
print('Pinned snapshots resolved; local paths remain private.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EVIDENCE = Path('evidence') / RUN_ID


## 3. Startup + smoke — stop here if it fails

One small-model 128/32 case, one measured trial after warmup. Inspect memory recovery and the timings before continuing. A successful smoke is **not** the completed lab. If interrupted, retain its JSON and restart only after understanding the failure.


In [ ]:
SMOKE = run_matrix(MODEL_PATHS, EVIDENCE / 'smoke', smoke=True)
print(json.dumps(summarize(SMOKE), indent=2))
assert SMOKE['status'] == 'complete'


## 4. Required matrix + matched batch comparison

Expect 50 raw trials, checkpointed after each trial. A new attempt needs a new RUN_ID; existing results cannot be overwritten. Do not change shape, dtype, attention or counts mid-run. The two models load sequentially and are released even on an exception. The matrix is deliberately small; it does not establish production capacity.


In [ ]:
assert SMOKE['status'] == 'complete', 'Pass the smoke first'
RUN = run_matrix(MODEL_PATHS, EVIDENCE / 'core')
SUMMARY = summarize(RUN)
checkpoint(EVIDENCE / 'topic02-summary.json', SUMMARY)
for row in SUMMARY:
    print(json.dumps(row))


## 5. CPU artifact contract

This checker rejects incomplete matrices, duplicated trials, wrong revisions, nonfinite timings and off-by-one/rate errors. Passing validates structure and arithmetic, **not authenticity or the correctness of your bottleneck conclusion**. It is also available in the public template as scripts/validate_topic02.py.


In [ ]:
#!/usr/bin/env python3
"""CPU-only structure/arithmetic check, not a proof that a benchmark is genuine."""
import argparse
import json
import math
from pathlib import Path

MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]


def validate(data):
    if (data.get("protocol"), data.get("status"), data.get("scope"), data.get("trials")) != (
            "topic02-phase-v2", "complete", "core", 5):
        raise ValueError("Expected a completed core protocol-v2 run with five trials; smoke/partial is not complete")
    if data.get("models") != MODELS or sorted(data.get("cases", [])) != sorted(map(list, CASES)):
        raise ValueError("Pinned model revisions or required cases differ")
    env = data.get("environment", {})
    if any(env.get(k) != v for k, v in {"dtype": "float16", "transformers": "4.51.3",
                                        "attention_requested": "sdpa", "logits_to_keep": 1}.items()):
        raise ValueError("Protocol runtime settings differ")
    if not all(env.get(k) for k in ("python", "torch", "cuda", "gpu")):
        raise ValueError("Missing environment provenance")
    expected = {(model, *case, repeat) for model in MODELS for case in CASES for repeat in range(5)}
    seen = set()
    for row in data.get("results", []):
        key = tuple(row[k] for k in ("model", "batch", "input_tokens", "output_tokens", "repeat"))
        if key not in expected or key in seen:
            raise ValueError("Unexpected or duplicate model/shape/trial")
        seen.add(key)
        model, batch, prompt, output, repeat = key
        for metric in ("prefill_wall_ms", "prefill_event_ms", "decode_wall_ms", "decode_event_ms",
                       "decode_aggregate_tokens_per_s", "decode_mean_step_ms", "peak_allocated_bytes",
                       "peak_reserved_bytes", "prefill_peak_allocated_bytes"):
            value = row.get(metric)
            if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{metric} must be a finite positive measurement")
        if (row.get("decode_steps_per_sequence"), row.get("generated_tokens"),
            row.get("retained_tokens_per_sequence")) != (output - 1, batch * output, prompt + output - 1):
            raise ValueError("Output/step/KV accounting mismatch")
        if not math.isclose(row["decode_aggregate_tokens_per_s"], batch * (output - 1) * 1000 / row["decode_wall_ms"], rel_tol=1e-7):
            raise ValueError("Aggregate decode rate uses the wrong numerator or clock")
        if not math.isclose(row["decode_mean_step_ms"], row["decode_wall_ms"] / (output - 1), rel_tol=1e-7):
            raise ValueError("Mean step interval uses the wrong denominator")
        if row["peak_reserved_bytes"] < row["peak_allocated_bytes"] or row["peak_allocated_bytes"] < row["prefill_peak_allocated_bytes"]:
            raise ValueError("Impossible allocated/reserved peak ordering")
    if seen != expected:
        raise ValueError(f"Missing trials: found {len(seen)} of {len(expected)}")
    return len(seen)



print('Validated raw trials:', validate(RUN))


## 6. Focused profiles — two shapes, not the entire matrix

The 0.5B model, P=128 and P=4096, B=1, O=8. Each trace has named prefill and seven-step decode ranges. Open the local Chrome traces in a trace viewer; do not upload private artifacts to a public service. Record the visible gaps and dominant operators separately for each phase. Requested SDPA is not proof of a particular FlashAttention kernel: identify the actual selected operator/kernel from the trace.

Profiling changes execution. Keep these traces separate from unprofiled timings. They cannot explain the 1.5B/128-output case directly, establish HBM traffic, or validate a production p95. Hardware-counter diagnosis with Nsight is guided self-study, not required for this core artifact.


In [ ]:
PROFILE = focused_profiles(MODEL_PATHS[list(MODELS)[0]], EVIDENCE / 'profiles')
print(json.dumps(PROFILE, indent=2))


## 7. Required analysis and hand-in

Keep this lab in your **private** course repository; follow the current round's tag/deadline procedure in the student guide. Do not change the leaderboard server contract just to run a notebook.

Submit **topic02-analysis.md**, the unmodified core **topic02-benchmark.json**, **topic02-summary.json**, both **topic02-trace-p128.json / topic02-trace-p4096.json**, and **topic02-profile-summary.json**. No model weights, caches, secrets or executed notebook outputs belong in the public template.

Your analysis must contain:

1. A manifest: model revisions, GPU, driver, runtime, dtype, attention path, exact cases and timing boundaries. Add the driver from nvidia-smi to your report; the notebook records the other runtime fields.
2. Three predictions written before the run; observed medians **and min/max plus all five raw trials**. Five samples are not credible p95.
3. Two matched input/output contrasts for each model and the B1→B4 128/32 comparison. Aggregate rate uses B×(O−1); per-sequence mean cadence uses decode wall time/(O−1). Do not equate either with service capacity.
4. A KV calculation from model.config and P+O−1 retained tokens, reconciled with allocated/reserved/peak bytes. Explain at least two omitted memory terms.
5. A roofline estimate with declared precision, memory boundary, FLOP convention and **source for machine bandwidth/compute**. A worked hypothetical roof is allowed if labeled; do not fabricate hardware counters.
6. Two trace observations (phase + prompt length + operator/range), an alternative explanation, one failed or unresolved prediction, and the smallest next discriminating test. A screenshot without interpretation is insufficient.
7. Safety/reproducibility: partial failures if any, resource peaks/recovery, cache reuse and exact run identifier. Never hide an OOM or silently shrink the required matrix.

**Success criterion:** a reviewer can reproduce your arithmetic and understand why the evidence supports—or fails to support—your hypothesis. A faster number alone is not the learning outcome. Optional extensions must be labeled separately and must retain their own raw evidence.
